# Notebook 165: テキスト生成の技術 — デコーディング戦略

| 項目 | 内容 |
|------|------|
| **難易度** | ★★★☆☆（中級） |
| **推定時間** | 90〜120分 |
| **カテゴリ** | 言語モデル |
| **前提知識** | Notebook 164（GPT自己回帰LM） |

---

## 学習目標

このノートブックを完了すると、以下ができるようになります：

- [ ] Greedy Search の実装と限界を説明できる
- [ ] Beam Search を実装し、ビーム幅の影響を可視化できる
- [ ] Top-k Sampling の確率分布の切り捨てを理解できる
- [ ] Nucleus (Top-p) Sampling の動的な切り捨てを実装できる
- [ ] Temperature のソフトマックスへの効果を実験できる
- [ ] 反復ペナルティ・長さペナルティを適用できる

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# セクション1: 環境セットアップ
# ============================================================
# 必要なライブラリのインポートと再現性の確保
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import defaultdict
import copy
import warnings
warnings.filterwarnings('ignore')

# --- 再現性の確保 ---
np.random.seed(42)
torch.manual_seed(42)

# --- 日本語フォント設定 ---
# macOS: 'Hiragino Sans', Linux: 'IPAGothic', Windows: 'MS Gothic'
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'Hiragino Sans'
elif platform.system() == 'Linux':
    plt.rcParams['font.family'] = 'IPAGothic'
else:
    plt.rcParams['font.family'] = 'MS Gothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# --- デバイス設定 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')
print(f'PyTorch バージョン: {torch.__version__}')
print(f'NumPy バージョン: {np.__version__}')
print('セットアップ完了！')

## 2. デコーディング問題の概要

### テキスト生成の核心

訓練済み言語モデルは、これまでの文脈 $w_1, w_2, \ldots, w_{t-1}$ が与えられたとき、
次のトークンの確率分布を出力します：

$$P(w_t \mid w_1, w_2, \ldots, w_{t-1})$$

この確率分布から**どのように次のトークンを選択するか**がデコーディング戦略です。

### 戦略の分類

| 種類 | 手法 | 特徴 |
|------|------|------|
| **決定的 (Deterministic)** | Greedy Search | 毎回最大確率のトークンを選ぶ |
| **決定的** | Beam Search | 複数候補を同時に探索 |
| **確率的 (Stochastic)** | Top-k Sampling | 上位k個からランダムに選ぶ |
| **確率的** | Nucleus (Top-p) Sampling | 累積確率pまでからランダムに選ぶ |
| **調整パラメータ** | Temperature | 分布の鋭さを調整 |
| **後処理** | 反復ペナルティ | 繰り返しを抑制 |

> **ポイント**: 決定的な手法は再現性が高いが多様性に欠ける。確率的な手法は多様だが品質が不安定になり得る。

## 3. シミュレーション用のモック言語モデル

In [ ]:
# ============================================================
# セクション3: シミュレーション用のモック言語モデル
# ============================================================
# 実際のLMの代わりに、確率分布を返すモック言語モデルを作成する。
# 語彙サイズ50の小さなモデルで、デコーディング戦略の動作を理解する。
# ============================================================

class MockLanguageModel:
    """
    シミュレーション用モック言語モデル。
    
    語彙は簡単な日本語単語で構成され、
    文脈に応じた next-token 確率分布を返す。
    実際の言語モデルの出力を模倣する。
    """
    def __init__(self, vocab_size=50, seed=42):
        # --- 語彙の定義 ---
        # 日本語の簡単な単語を語彙として使用
        self.vocab = [
            '私', 'は', 'が', 'の', 'を', 'に', 'で', 'と', 'も', 'から',
            '今日', '明日', '昨日', '天気', '良い', '悪い', '普通', '最高', '素晴らしい', '美しい',
            '食べ', 'た', 'ます', 'です', 'した', 'する', 'なる', 'ある', 'いる', 'できる',
            '猫', '犬', '鳥', '魚', '花', '空', '海', '山', '川', '森',
            '大きい', '小さい', '新しい', '古い', '赤い', '青い', '白い', '黒い', '。', '、'
        ]
        self.vocab_size = len(self.vocab)
        # トークンID → 単語 の辞書
        self.id2word = {i: w for i, w in enumerate(self.vocab)}
        # 単語 → トークンID の辞書
        self.word2id = {w: i for i, w in enumerate(self.vocab)}
        
        # --- 簡単な遷移確率行列（文脈依存の確率を模倣）---
        np.random.seed(seed)
        # ランダムなロジットを生成し、一部のパターンを強化する
        self.base_logits = np.random.randn(vocab_size, vocab_size).astype(np.float32)
        
        # 自然な遷移を強化：例）「私」→「は」、「天気」→「が」など
        self._add_transition('私', 'は', 3.0)
        self._add_transition('は', '今日', 2.0)
        self._add_transition('今日', 'の', 2.5)
        self._add_transition('の', '天気', 2.5)
        self._add_transition('天気', 'は', 2.0)
        self._add_transition('天気', 'が', 2.5)
        self._add_transition('が', '良い', 2.0)
        self._add_transition('が', '素晴らしい', 1.8)
        self._add_transition('が', '美しい', 1.5)
        self._add_transition('良い', 'です', 2.5)
        self._add_transition('素晴らしい', 'です', 2.0)
        self._add_transition('です', '。', 3.0)
        self._add_transition('。', '私', 1.5)
        self._add_transition('。', '今日', 1.5)
        self._add_transition('。', '明日', 1.5)
        self._add_transition('猫', 'が', 2.0)
        self._add_transition('犬', 'が', 2.0)
        self._add_transition('空', 'が', 2.0)
        self._add_transition('空', 'は', 1.5)
        self._add_transition('美しい', 'です', 2.0)
        self._add_transition('食べ', 'た', 2.5)
        self._add_transition('食べ', 'ます', 2.0)
    
    def _add_transition(self, from_word, to_word, bonus):
        """特定の遷移にボーナスを追加する"""
        if from_word in self.word2id and to_word in self.word2id:
            i = self.word2id[from_word]
            j = self.word2id[to_word]
            self.base_logits[i, j] += bonus
    
    def get_logits(self, token_ids):
        """
        トークンIDの列を受け取り、次トークンのロジットを返す。
        最後のトークンに基づいて遷移確率を返す（簡略化のためバイグラムモデル）。
        
        Parameters:
            token_ids: list[int] — これまでに生成されたトークンIDのリスト
        Returns:
            logits: torch.Tensor — shape (vocab_size,)
        """
        if len(token_ids) == 0:
            # 文頭の場合はユニフォームに近いロジット
            logits = torch.randn(self.vocab_size)
            # 「私」「今日」「猫」などの文頭トークンを強化
            for w in ['私', '今日', '明日', '昨日', '猫', '犬', '空']:
                logits[self.word2id[w]] += 2.0
            return logits
        
        last_token = token_ids[-1]
        logits = torch.tensor(self.base_logits[last_token], dtype=torch.float32)
        
        # 少しノイズを加えて多様性を出す
        logits += torch.randn(self.vocab_size) * 0.1
        return logits
    
    def decode(self, token_ids):
        """トークンIDの列をテキストに変換する"""
        return ''.join([self.id2word[tid] for tid in token_ids])

# --- モデルのインスタンス化と動作確認 ---
lm = MockLanguageModel()

print('=== モック言語モデルの動作確認 ===')
print(f'語彙サイズ: {lm.vocab_size}')
print(f'語彙の最初の10個: {lm.vocab[:10]}')
print()

# 「私」の次のトークンの確率分布を確認
token_ids = [lm.word2id['私']]
logits = lm.get_logits(token_ids)
probs = F.softmax(logits, dim=-1)

# 上位5個のトークンを表示
top5_probs, top5_ids = torch.topk(probs, 5)
print('「私」の次に来やすいトークン Top-5:')
for prob, tid in zip(top5_probs, top5_ids):
    print(f'  {lm.id2word[tid.item()]:6s}  確率: {prob.item():.4f}')

## 4. Greedy Search

### 概要

Greedy Search（貪欲探索）は最もシンプルなデコーディング戦略です。
各ステップで**最も確率が高いトークン**を選択します。

$$w_t = \arg\max_w P(w \mid w_1, \ldots, w_{t-1})$$

**メリット**: 実装が簡単、高速、決定的（再現性あり）

**デメリット**: 局所最適に陥りやすい、繰り返しが発生しやすい

In [ ]:
# ============================================================
# セクション4: Greedy Search の実装
# ============================================================
# 毎ステップで最も確率が高いトークンを選択する。
# シンプルだが、局所最適に陥りやすいという問題がある。
# ============================================================

def greedy_search(model, start_tokens, max_length=15):
    """
    Greedy Search によるテキスト生成。
    
    Parameters:
        model: MockLanguageModel — 言語モデル
        start_tokens: list[int] — 開始トークンID列
        max_length: int — 最大生成長
    Returns:
        generated: list[int] — 生成されたトークンID列
        log_prob: float — 合計対数確率
    """
    generated = list(start_tokens)
    total_log_prob = 0.0
    
    for step in range(max_length):
        # --- ロジットを取得 ---
        logits = model.get_logits(generated)
        # --- ソフトマックスで確率に変換 ---
        probs = F.softmax(logits, dim=-1)
        # --- 最も確率が高いトークンを選択（貪欲選択） ---
        best_id = torch.argmax(probs).item()
        best_prob = probs[best_id].item()
        
        # --- 対数確率を蓄積 ---
        total_log_prob += np.log(best_prob + 1e-10)
        
        generated.append(best_id)
        
        # 句点で終了
        if model.id2word[best_id] == '。':
            break
    
    return generated, total_log_prob

# --- Greedy Search の実行 ---
# 「私」から生成を開始
start = [lm.word2id['私']]
greedy_result, greedy_log_prob = greedy_search(lm, start, max_length=15)

print('=== Greedy Search の結果 ===')
print(f'生成テキスト: {lm.decode(greedy_result)}')
print(f'トークン列: {[lm.id2word[t] for t in greedy_result]}')
print(f'合計対数確率: {greedy_log_prob:.4f}')
print()

# --- 問題点の実証: 繰り返しが発生するケース ---
# 長めに生成すると反復が見られることがある
long_result, _ = greedy_search(lm, start, max_length=30)
print('=== Greedy Search（長文生成）===')
print(f'生成テキスト: {lm.decode(long_result)}')
print(f'トークン数: {len(long_result)}')

# トークンの繰り返しを検出
from collections import Counter
token_counts = Counter(long_result)
print('\n繰り返しの多いトークン:')
for tid, count in token_counts.most_common(5):
    print(f'  「{lm.id2word[tid]}」: {count}回')

In [ ]:
# ============================================================
# セクション4（続き）: Greedy Search の可視化
# ============================================================
# 各ステップでの確率分布と選択されたトークンを可視化する。
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Greedy Search: 各ステップでの確率分布と選択', fontsize=14, fontweight='bold')

# 最初の4ステップを可視化
current_tokens = list(start)

for step, ax in enumerate(axes.flat):
    logits = lm.get_logits(current_tokens)
    probs = F.softmax(logits, dim=-1).numpy()
    
    # 上位10個を表示
    top_k_indices = np.argsort(probs)[-10:][::-1]
    top_k_probs = probs[top_k_indices]
    top_k_words = [lm.id2word[i] for i in top_k_indices]
    
    # 選択されたトークン（最大確率）をハイライト
    colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(top_k_words))]
    
    bars = ax.bar(range(len(top_k_words)), top_k_probs, color=colors)
    ax.set_xticks(range(len(top_k_words)))
    ax.set_xticklabels(top_k_words, fontsize=10)
    ax.set_ylabel('確率')
    
    context_text = ''.join([lm.id2word[t] for t in current_tokens])
    ax.set_title(f'Step {step+1}: 文脈「{context_text}」', fontsize=11)
    ax.set_ylim(0, max(top_k_probs) * 1.3)
    
    # 選択されたトークンを注釈
    chosen = top_k_words[0]
    ax.annotate(f'選択: 「{chosen}」', xy=(0, top_k_probs[0]),
                xytext=(3, top_k_probs[0] * 1.1),
                fontsize=9, color='#e74c3c', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#e74c3c'))
    
    # 次のステップ用に選択されたトークンを追加
    best_id = top_k_indices[0]
    current_tokens.append(best_id)

plt.tight_layout()
plt.show()

print('赤色 = Greedy で選択されたトークン（最大確率）')
print('青色 = 選択されなかった候補トークン')

## 5. Beam Search

### 概要

Beam Search は、各ステップで**ビーム幅 k 個の候補シーケンス**を同時に保持する探索手法です。
Greedy Search の一般化であり（k=1 のとき Greedy と同等）、より良い全体最適解を見つけやすくなります。

$$\text{Score}(y_{1:t}) = \sum_{i=1}^{t} \log P(y_i \mid y_{<i})$$

長さペナルティ付き：

$$\text{Score}(y_{1:t}) = \frac{1}{t^\alpha} \sum_{i=1}^{t} \log P(y_i \mid y_{<i})$$

In [ ]:
# ============================================================
# セクション5: Beam Search の実装
# ============================================================
# ビーム幅 k の候補を保持しながら探索を進める。
# k=1 は Greedy Search と同等になる。
# ============================================================

def beam_search(model, start_tokens, beam_width=3, max_length=15, length_penalty_alpha=0.0):
    """
    Beam Search によるテキスト生成。
    
    Parameters:
        model: MockLanguageModel — 言語モデル
        start_tokens: list[int] — 開始トークンID列
        beam_width: int — ビーム幅（候補数）
        max_length: int — 最大生成長
        length_penalty_alpha: float — 長さペナルティの強さ（0で無効）
    Returns:
        best_sequence: list[int] — 最良のトークンID列
        best_score: float — 最良のスコア
        all_beams_history: list — 各ステップのビーム状態（可視化用）
    """
    # 各ビーム: (トークン列, 累積対数確率)
    beams = [(list(start_tokens), 0.0)]
    completed = []  # 完了したシーケンス
    all_beams_history = []  # 可視化用の履歴
    
    for step in range(max_length):
        all_candidates = []
        
        for seq, score in beams:
            # --- ロジットを取得 ---
            logits = model.get_logits(seq)
            probs = F.softmax(logits, dim=-1)
            log_probs = torch.log(probs + 1e-10)
            
            # --- 上位 beam_width * 2 個の候補を取得 ---
            top_log_probs, top_ids = torch.topk(log_probs, min(beam_width * 2, model.vocab_size))
            
            for lp, tid in zip(top_log_probs, top_ids):
                new_seq = seq + [tid.item()]
                new_score = score + lp.item()
                all_candidates.append((new_seq, new_score))
        
        # --- スコアでソートして上位 beam_width 個を保持 ---
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        
        beams = []
        for seq, score in all_candidates:
            if model.id2word[seq[-1]] == '。':
                # 長さペナルティを適用
                length = len(seq) - len(start_tokens)
                if length_penalty_alpha > 0:
                    penalized_score = score / (length ** length_penalty_alpha)
                else:
                    penalized_score = score
                completed.append((seq, penalized_score))
            else:
                beams.append((seq, score))
            
            if len(beams) >= beam_width:
                break
        
        # 履歴を保存（可視化用）
        all_beams_history.append([
            (list(seq), score) for seq, score in beams[:beam_width]
        ])
        
        if len(beams) == 0:
            break
    
    # --- 最良のシーケンスを選択 ---
    # 完了したものと未完了のものから最良を選ぶ
    all_results = completed + beams
    if length_penalty_alpha > 0:
        # 未完了のものにもペナルティを適用
        final_results = []
        for seq, score in all_results:
            length = len(seq) - len(start_tokens)
            penalized = score / (max(length, 1) ** length_penalty_alpha)
            final_results.append((seq, penalized))
        all_results = final_results
    
    all_results.sort(key=lambda x: x[1], reverse=True)
    best_seq, best_score = all_results[0]
    
    return best_seq, best_score, all_beams_history

# --- ビーム幅を変えて生成結果を比較 ---
start = [lm.word2id['私']]

print('=== Beam Search: ビーム幅の比較 ===')
print()

beam_results = {}
for bw in [1, 3, 5, 10]:
    torch.manual_seed(42)
    np.random.seed(42)
    result, score, history = beam_search(lm, start, beam_width=bw, max_length=15)
    beam_results[bw] = (result, score)
    text = lm.decode(result)
    print(f'ビーム幅 {bw:2d}: スコア={score:.4f}  テキスト=「{text}」')

print()
print('→ ビーム幅が大きいほど、より高スコアの列が見つかる傾向がある。')

In [ ]:
# ============================================================
# セクション5（続き）: Beam Search の木構造の可視化
# ============================================================
# ビーム探索の各ステップでの候補を木構造的に可視化する。
# ============================================================

# ビーム幅3で再度実行し、履歴を可視化
torch.manual_seed(42)
np.random.seed(42)
_, _, beam_history = beam_search(lm, start, beam_width=3, max_length=8)

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_title('Beam Search の探索過程（ビーム幅=3）', fontsize=14, fontweight='bold')

# 各ステップの各ビームを描画
max_steps = min(len(beam_history), 7)
colors_beam = ['#e74c3c', '#3498db', '#2ecc71']

for step_idx in range(max_steps):
    beams_at_step = beam_history[step_idx]
    n_beams = min(len(beams_at_step), 3)
    
    for beam_idx in range(n_beams):
        seq, score = beams_at_step[beam_idx]
        # テキストを生成
        text = lm.decode(seq)
        if len(text) > 12:
            text = '...' + text[-10:]
        
        # 位置を計算
        x = step_idx
        y = (n_beams - 1) / 2 - beam_idx  # 中央揃え
        
        # ノードを描画
        color = colors_beam[beam_idx % 3]
        ax.plot(x, y, 'o', markersize=12, color=color, zorder=5)
        ax.annotate(f'{text}\n({score:.2f})',
                   xy=(x, y), xytext=(x, y - 0.35),
                   fontsize=7, ha='center', va='top',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.2))
        
        # 前のステップとの接続線
        if step_idx > 0:
            prev_beams = beam_history[step_idx - 1]
            prev_n = min(len(prev_beams), 3)
            for prev_idx in range(prev_n):
                prev_y = (prev_n - 1) / 2 - prev_idx
                ax.plot([step_idx - 1, step_idx], [prev_y, y],
                       '-', color='gray', alpha=0.2, linewidth=0.5)

ax.set_xlabel('ステップ', fontsize=12)
ax.set_ylabel('ビーム候補', fontsize=12)
ax.set_xticks(range(max_steps))
ax.set_xticklabels([f'Step {i+1}' for i in range(max_steps)])
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

print('各ノードのラベル: テキスト（累積対数確率スコア）')
print('色は異なるビーム候補を表す。')

In [ ]:
# ============================================================
# セクション5（続き）: 長さペナルティの効果
# ============================================================
# 長さペナルティを適用すると、短いシーケンスが不当に有利になる
# 問題を軽減できる。
# ============================================================

print('=== 長さペナルティの効果 ===')
print()

for alpha in [0.0, 0.3, 0.6, 1.0]:
    torch.manual_seed(42)
    np.random.seed(42)
    result, score, _ = beam_search(lm, start, beam_width=5, max_length=15,
                                    length_penalty_alpha=alpha)
    text = lm.decode(result)
    print(f'α={alpha:.1f}: スコア={score:.4f}  長さ={len(result)}  テキスト=「{text}」')

print()
print('→ α>0 にすると、長いシーケンスが不当に低スコアになるのを防ぐ。')

## 6. Temperature Scaling

### 概要

Temperature（温度）パラメータ $T$ は、ソフトマックスの出力分布の**鋭さ**を制御します：

$$P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

| 温度 $T$ | 効果 |
|-----------|------|
| $T \to 0$ | 分布が鋭くなる（ほぼ Greedy） |
| $T = 1$ | 元の分布（変化なし） |
| $T > 1$ | 分布が平坦になる（より多様） |
| $T \to \infty$ | 一様分布に近づく |

In [ ]:
# ============================================================
# セクション6: Temperature Scaling の実装と可視化
# ============================================================
# ソフトマックスの温度パラメータが確率分布に与える影響を
# 実験的に確認する。
# ============================================================

def apply_temperature(logits, temperature):
    """
    ロジットに温度スケーリングを適用する。
    
    Parameters:
        logits: torch.Tensor — 元のロジット
        temperature: float — 温度パラメータ（>0）
    Returns:
        probs: torch.Tensor — 温度適用後の確率分布
    """
    # 温度が0に近い場合はargmaxに相当
    if temperature < 1e-6:
        probs = torch.zeros_like(logits)
        probs[torch.argmax(logits)] = 1.0
        return probs
    
    # ロジットを温度で割ってからソフトマックス
    scaled_logits = logits / temperature
    probs = F.softmax(scaled_logits, dim=-1)
    return probs

# --- 温度による分布変化を可視化 ---
# 「私は今日の天気」の後の確率分布を取得
context = [lm.word2id[w] for w in ['天気']]
logits = lm.get_logits(context)

# 上位15個のトークンのみ表示
top15_vals, top15_ids = torch.topk(logits, 15)
top15_words = [lm.id2word[i.item()] for i in top15_ids]

temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]
colors_temp = ['#c0392b', '#e67e22', '#27ae60', '#2980b9', '#8e44ad']

fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=False)
fig.suptitle('Temperature Scaling: 温度による確率分布の変化', fontsize=14, fontweight='bold')

for idx, (temp, color) in enumerate(zip(temperatures, colors_temp)):
    ax = axes[idx]
    probs = apply_temperature(logits, temp)
    
    # 上位15個の確率を取得
    top15_probs = probs[top15_ids].detach().numpy()
    
    ax.barh(range(len(top15_words)), top15_probs[::-1], color=color, alpha=0.8)
    ax.set_yticks(range(len(top15_words)))
    ax.set_yticklabels(top15_words[::-1], fontsize=9)
    ax.set_xlabel('確率')
    ax.set_title(f'T = {temp}', fontsize=12, fontweight='bold')
    
    # エントロピーを計算して表示
    p = probs.detach().numpy()
    entropy = -np.sum(p * np.log(p + 1e-10))
    ax.text(0.95, 0.05, f'H = {entropy:.2f}',
           transform=ax.transAxes, fontsize=9, ha='right',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print('H = エントロピー（情報量）: 高いほど分布が平坦で不確実性が高い')
print('T→0: ほぼ決定的（Greedy相当）, T=1: 元の分布, T>1: より多様')

In [ ]:
# ============================================================
# セクション6（続き）: Temperature による生成結果の比較
# ============================================================
# 温度を変えてサンプリング生成し、結果を比較する。
# ============================================================

def sample_with_temperature(model, start_tokens, temperature=1.0, max_length=15):
    """
    Temperature Sampling によるテキスト生成。
    
    Parameters:
        model: MockLanguageModel — 言語モデル
        start_tokens: list[int] — 開始トークンID列
        temperature: float — 温度パラメータ
        max_length: int — 最大生成長
    Returns:
        generated: list[int] — 生成されたトークンID列
    """
    generated = list(start_tokens)
    
    for step in range(max_length):
        logits = model.get_logits(generated)
        probs = apply_temperature(logits, temperature)
        
        # 確率分布からサンプリング
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_token)
        
        if model.id2word[next_token] == '。':
            break
    
    return generated

print('=== Temperature による生成結果の比較 ===')
print('（同じシードで3回ずつ生成）')
print()

for temp in [0.1, 0.5, 1.0, 1.5, 2.0]:
    print(f'--- Temperature = {temp} ---')
    for trial in range(3):
        torch.manual_seed(42 + trial)
        result = sample_with_temperature(lm, start, temperature=temp, max_length=12)
        print(f'  試行{trial+1}: {lm.decode(result)}')
    print()

## 7. Top-k Sampling

### 概要

Top-k Sampling は、確率の高い**上位 k 個のトークンだけ**を候補として残し、
それ以外を確率0にしてから再正規化してサンプリングする手法です。

$$P'(w_i) = \begin{cases} \frac{P(w_i)}{\sum_{j \in \text{top-k}} P(w_j)} & \text{if } w_i \in \text{top-k} \\ 0 & \text{otherwise} \end{cases}$$

**利点**: 低確率のノイズトークンを除外できる

**課題**: k は固定値なので、分布の形状によっては適切でないことがある

In [ ]:
# ============================================================
# セクション7: Top-k Sampling の実装
# ============================================================
# 上位k個のトークンのみを残し、それ以外を除外する。
# kの値によって多様性と品質のトレードオフが変わる。
# ============================================================

def top_k_filtering(logits, k):
    """
    Top-k フィルタリングを適用する。
    
    Parameters:
        logits: torch.Tensor — 元のロジット
        k: int — 保持するトークン数
    Returns:
        filtered_probs: torch.Tensor — フィルタリング後の確率分布
    """
    if k <= 0 or k >= len(logits):
        return F.softmax(logits, dim=-1)
    
    # 上位k個以外のロジットを-infに設定
    top_k_values, top_k_indices = torch.topk(logits, k)
    
    # -inf で埋めたロジットを作成
    filtered_logits = torch.full_like(logits, float('-inf'))
    filtered_logits[top_k_indices] = top_k_values
    
    # ソフトマックスで正規化（-inf は 0 になる）
    filtered_probs = F.softmax(filtered_logits, dim=-1)
    return filtered_probs


def top_k_sampling(model, start_tokens, k=10, temperature=1.0, max_length=15):
    """
    Top-k Sampling によるテキスト生成。
    
    Parameters:
        model: MockLanguageModel
        start_tokens: list[int]
        k: int — 上位k個を残す
        temperature: float — 温度
        max_length: int
    Returns:
        generated: list[int]
    """
    generated = list(start_tokens)
    
    for step in range(max_length):
        logits = model.get_logits(generated)
        # 温度スケーリングを先に適用
        if temperature != 1.0:
            logits = logits / temperature
        # Top-k フィルタリング
        probs = top_k_filtering(logits, k)
        # サンプリング
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_token)
        
        if model.id2word[next_token] == '。':
            break
    
    return generated

# --- k の値を変えて生成結果を比較 ---
print('=== Top-k Sampling: k の比較 ===')
print()

for k in [1, 5, 10, 50]:
    print(f'--- k = {k} ---')
    for trial in range(3):
        torch.manual_seed(42 + trial)
        result = top_k_sampling(lm, start, k=k, max_length=12)
        print(f'  試行{trial+1}: {lm.decode(result)}')
    print()

print('→ k=1 は Greedy と同等、k が大きいほど多様な生成が可能')

In [ ]:
# ============================================================
# セクション7（続き）: Top-k フィルタの効果を可視化
# ============================================================
# 元の確率分布と Top-k フィルタ後の分布を比較する。
# ============================================================

context = [lm.word2id['天気']]
logits = lm.get_logits(context)
orig_probs = F.softmax(logits, dim=-1).detach().numpy()

# 確率順にソートしたインデックス
sorted_indices = np.argsort(orig_probs)[::-1]
sorted_probs = orig_probs[sorted_indices]
sorted_words = [lm.id2word[i] for i in sorted_indices]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Top-k Sampling: フィルタリングの効果', fontsize=14, fontweight='bold')

for idx, k in enumerate([3, 5, 10, 20]):
    ax = axes[idx // 2, idx % 2]
    
    # Top-k フィルタリング後の確率
    filtered_probs = top_k_filtering(logits, k).detach().numpy()
    filtered_sorted = filtered_probs[sorted_indices]
    
    n_show = min(25, len(sorted_words))
    x = range(n_show)
    
    # 元の確率（薄い色）
    ax.bar(x, sorted_probs[:n_show], alpha=0.3, color='gray', label='元の分布')
    # フィルタ後の確率（濃い色）
    ax.bar(x, filtered_sorted[:n_show], alpha=0.8, color='#e74c3c', label=f'Top-{k}')
    
    # k の境界線
    if k < n_show:
        ax.axvline(x=k - 0.5, color='black', linestyle='--', alpha=0.5, label=f'k={k} 境界')
    
    ax.set_xticks(x)
    ax.set_xticklabels(sorted_words[:n_show], rotation=45, fontsize=8, ha='right')
    ax.set_ylabel('確率')
    ax.set_title(f'Top-k (k={k})', fontsize=12)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('灰色 = 元の確率分布, 赤色 = Top-k フィルタ後の確率分布')
print('点線 = k の境界（これより右のトークンは除外される）')

## 8. Nucleus (Top-p) Sampling

### 概要

Nucleus Sampling（Top-p Sampling）は、確率を高い順に並べたとき、
**累積確率が p を超えるまで**のトークンを候補に残す手法です。

$$V^{(p)} = \min \left\{ V' \subseteq V : \sum_{w \in V'} P(w) \geq p \right\}$$

Top-k と異なり、候補数が**分布の形状に応じて動的に変化**します。

- 分布が鋭い（確信度高い）→ 少数のトークンだけ残る
- 分布が平坦（不確実）→ 多くのトークンが残る

この動的な性質が Top-p の大きな利点です。

In [ ]:
# ============================================================
# セクション8: Nucleus (Top-p) Sampling の実装
# ============================================================
# 累積確率が p を超えるまでのトークンを保持する。
# 分布の形状に応じて候補数が動的に変化するのが特徴。
# ============================================================

def top_p_filtering(logits, p):
    """
    Top-p (Nucleus) フィルタリングを適用する。
    
    Parameters:
        logits: torch.Tensor — 元のロジット
        p: float — 累積確率の閾値 (0 < p <= 1)
    Returns:
        filtered_probs: torch.Tensor — フィルタリング後の確率分布
    """
    probs = F.softmax(logits, dim=-1)
    
    # 確率を降順にソート
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    
    # 累積確率を計算
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # 累積確率が p を超える最初の位置を見つける
    # ただし、最初のトークンは必ず残す
    sorted_mask = cumulative_probs - sorted_probs >= p  # shift right by 1
    
    # マスクされたトークンの確率を0にする
    sorted_probs[sorted_mask] = 0.0
    
    # 元のインデックス順に戻す
    filtered_probs = torch.zeros_like(probs)
    filtered_probs[sorted_indices] = sorted_probs
    
    # 再正規化
    filtered_probs = filtered_probs / filtered_probs.sum()
    
    return filtered_probs


def top_p_sampling(model, start_tokens, p=0.9, temperature=1.0, max_length=15):
    """
    Nucleus (Top-p) Sampling によるテキスト生成。
    
    Parameters:
        model: MockLanguageModel
        start_tokens: list[int]
        p: float — 累積確率の閾値
        temperature: float — 温度
        max_length: int
    Returns:
        generated: list[int]
    """
    generated = list(start_tokens)
    
    for step in range(max_length):
        logits = model.get_logits(generated)
        # 温度スケーリング
        if temperature != 1.0:
            logits = logits / temperature
        # Top-p フィルタリング
        probs = top_p_filtering(logits, p)
        # サンプリング
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_token)
        
        if model.id2word[next_token] == '。':
            break
    
    return generated

# --- p の値を変えて生成結果を比較 ---
print('=== Nucleus (Top-p) Sampling: p の比較 ===')
print()

for p_val in [0.1, 0.5, 0.9, 0.95]:
    print(f'--- p = {p_val} ---')
    for trial in range(3):
        torch.manual_seed(42 + trial)
        result = top_p_sampling(lm, start, p=p_val, max_length=12)
        print(f'  試行{trial+1}: {lm.decode(result)}')
    print()

print('→ p が小さいほど保守的（少数候補）、p が大きいほど多様（多数候補）')

In [ ]:
# ============================================================
# セクション8（続き）: Top-p の累積確率と切断点の可視化
# ============================================================
# 累積確率分布を描画し、p の値による切断点を示す。
# Top-k との違い（動的 vs 固定）を明確にする。
# ============================================================

# 2つの異なる分布（鋭い分布と平坦な分布）で比較
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top-p Sampling: 累積確率と切断点', fontsize=14, fontweight='bold')

# --- 分布1: 鋭い分布（「です」の後 → 「。」が非常に高確率）---
context_sharp = [lm.word2id['です']]
logits_sharp = lm.get_logits(context_sharp)
probs_sharp = F.softmax(logits_sharp, dim=-1)
sorted_probs_sharp, sorted_idx_sharp = torch.sort(probs_sharp, descending=True)
cum_probs_sharp = torch.cumsum(sorted_probs_sharp, dim=-1).detach().numpy()
sorted_words_sharp = [lm.id2word[i.item()] for i in sorted_idx_sharp]

# --- 分布2: 平坦な分布（「が」の後 → 複数の形容詞が候補）---
context_flat = [lm.word2id['が']]
logits_flat = lm.get_logits(context_flat)
probs_flat = F.softmax(logits_flat, dim=-1)
sorted_probs_flat, sorted_idx_flat = torch.sort(probs_flat, descending=True)
cum_probs_flat = torch.cumsum(sorted_probs_flat, dim=-1).detach().numpy()
sorted_words_flat = [lm.id2word[i.item()] for i in sorted_idx_flat]

n_show = 20
p_values = [0.5, 0.9]
p_colors = ['#e74c3c', '#3498db']

for ax_idx, (cum_probs, sorted_words, title) in enumerate([
    (cum_probs_sharp, sorted_words_sharp, '鋭い分布: 文脈「です」の後'),
    (cum_probs_flat, sorted_words_flat, '平坦な分布: 文脈「が」の後')
]):
    ax = axes[ax_idx]
    
    # 累積確率をプロット
    ax.bar(range(n_show), cum_probs[:n_show], color='#95a5a6', alpha=0.6, label='累積確率')
    
    # p の水平線と切断点
    for p_val, p_col in zip(p_values, p_colors):
        ax.axhline(y=p_val, color=p_col, linestyle='--', alpha=0.8, label=f'p={p_val}')
        # 切断点を見つける
        cutoff = np.searchsorted(cum_probs[:n_show], p_val)
        if cutoff < n_show:
            ax.axvline(x=cutoff + 0.5, color=p_col, linestyle=':', alpha=0.5)
            ax.annotate(f'{cutoff+1}個', xy=(cutoff + 0.5, p_val),
                       xytext=(cutoff + 2, p_val - 0.08),
                       fontsize=10, color=p_col, fontweight='bold',
                       arrowprops=dict(arrowstyle='->', color=p_col))
    
    ax.set_xticks(range(n_show))
    ax.set_xticklabels(sorted_words[:n_show], rotation=45, fontsize=8, ha='right')
    ax.set_ylabel('累積確率')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print('【Top-k vs Top-p の違い】')
print('Top-k: 常に固定のk個を候補にする → 分布の形状を考慮しない')
print('Top-p: 累積確率に基づき動的に候補数が変わる → 分布の形状に適応')
print('  鋭い分布 → 少数のトークンで p に到達 → 少数候補')
print('  平坦な分布 → 多数のトークンが必要 → 多数候補')

## 9. 反復ペナルティ

### 概要

テキスト生成でよくある問題が**同じフレーズの繰り返し**です。
反復ペナルティは、すでに生成されたトークンの確率を下げることでこれを防ぎます。

| ペナルティの種類 | 数式 | 特徴 |
|-----------------|------|------|
| **頻度ペナルティ** | $\text{logit}_i \leftarrow \text{logit}_i - \alpha \cdot \text{count}(i)$ | 出現回数に比例して減少 |
| **プレゼンスペナルティ** | $\text{logit}_i \leftarrow \text{logit}_i - \beta \cdot \mathbb{1}[\text{count}(i) > 0]$ | 出現したかどうかのみ（回数不問） |

In [ ]:
# ============================================================
# セクション9: 反復ペナルティの実装
# ============================================================
# すでに生成したトークンのロジットにペナルティを適用して、
# 繰り返しを抑制する。頻度ペナルティとプレゼンスペナルティの
# 2種類を実装する。
# ============================================================

def apply_repetition_penalty(logits, generated_tokens, 
                              frequency_penalty=0.0, 
                              presence_penalty=0.0):
    """
    反復ペナルティを適用する。
    
    Parameters:
        logits: torch.Tensor — 元のロジット
        generated_tokens: list[int] — これまでに生成されたトークン
        frequency_penalty: float — 頻度ペナルティの強さ
        presence_penalty: float — プレゼンスペナルティの強さ
    Returns:
        penalized_logits: torch.Tensor — ペナルティ適用後のロジット
    """
    penalized_logits = logits.clone()
    
    if len(generated_tokens) == 0:
        return penalized_logits
    
    # トークンの出現回数をカウント
    token_counts = defaultdict(int)
    for tid in generated_tokens:
        token_counts[tid] += 1
    
    for tid, count in token_counts.items():
        # 頻度ペナルティ: 出現回数に比例
        penalized_logits[tid] -= frequency_penalty * count
        # プレゼンスペナルティ: 出現したら一律減算
        if count > 0:
            penalized_logits[tid] -= presence_penalty
    
    return penalized_logits


def sample_with_penalties(model, start_tokens, max_length=20,
                          temperature=1.0, top_p=0.9,
                          frequency_penalty=0.0, presence_penalty=0.0):
    """
    反復ペナルティ付きサンプリング。
    
    Parameters:
        model: MockLanguageModel
        start_tokens: list[int]
        max_length: int
        temperature: float
        top_p: float
        frequency_penalty: float
        presence_penalty: float
    Returns:
        generated: list[int]
    """
    generated = list(start_tokens)
    
    for step in range(max_length):
        logits = model.get_logits(generated)
        
        # --- 反復ペナルティを適用 ---
        logits = apply_repetition_penalty(
            logits, generated[len(start_tokens):],  # 生成部分のみカウント
            frequency_penalty=frequency_penalty,
            presence_penalty=presence_penalty
        )
        
        # --- 温度スケーリング ---
        if temperature != 1.0:
            logits = logits / temperature
        
        # --- Top-p フィルタリング ---
        probs = top_p_filtering(logits, top_p)
        
        # --- サンプリング ---
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_token)
        
        if model.id2word[next_token] == '。':
            break
    
    return generated

# --- ペナルティの効果を比較 ---
print('=== 反復ペナルティの効果 ===')
print()

configs = [
    ('ペナルティなし', 0.0, 0.0),
    ('頻度ペナルティ=1.0', 1.0, 0.0),
    ('頻度ペナルティ=2.0', 2.0, 0.0),
    ('プレゼンスペナルティ=1.0', 0.0, 1.0),
    ('両方 (freq=1.0, pres=0.5)', 1.0, 0.5),
]

for name, freq_p, pres_p in configs:
    print(f'--- {name} ---')
    for trial in range(2):
        torch.manual_seed(42 + trial)
        result = sample_with_penalties(
            lm, start, max_length=20,
            temperature=0.8, top_p=0.9,
            frequency_penalty=freq_p,
            presence_penalty=pres_p
        )
        tokens = [lm.id2word[t] for t in result]
        print(f'  試行{trial+1}: {lm.decode(result)}')
        # ユニークトークン率
        gen_tokens = result[len(start):]
        unique_ratio = len(set(gen_tokens)) / max(len(gen_tokens), 1)
        print(f'         ユニーク率: {unique_ratio:.2f}')
    print()

## 10. 全手法の比較実験

In [ ]:
# ============================================================
# セクション10: 全手法の比較実験
# ============================================================
# 同じプロンプトで全てのデコーディング戦略を適用し、
# 生成結果を並列比較する。
# ============================================================

print('=' * 70)
print('全手法の比較実験')
print('開始トークン: 「私」 | 最大生成長: 12')
print('=' * 70)
print()

results_comparison = {}

# --- 1. Greedy Search ---
torch.manual_seed(42)
np.random.seed(42)
result, score, _ = beam_search(lm, start, beam_width=1, max_length=12)
results_comparison['Greedy'] = lm.decode(result)
print(f'Greedy Search:       {lm.decode(result)}')

# --- 2. Beam Search (k=5) ---
torch.manual_seed(42)
np.random.seed(42)
result, score, _ = beam_search(lm, start, beam_width=5, max_length=12)
results_comparison['Beam(k=5)'] = lm.decode(result)
print(f'Beam Search (k=5):   {lm.decode(result)}')

# --- 3. Temperature Sampling (T=0.7) ---
torch.manual_seed(42)
result = sample_with_temperature(lm, start, temperature=0.7, max_length=12)
results_comparison['Temp(0.7)'] = lm.decode(result)
print(f'Temperature (T=0.7): {lm.decode(result)}')

# --- 4. Top-k Sampling (k=10) ---
torch.manual_seed(42)
result = top_k_sampling(lm, start, k=10, max_length=12)
results_comparison['Top-k(10)'] = lm.decode(result)
print(f'Top-k (k=10):        {lm.decode(result)}')

# --- 5. Nucleus Sampling (p=0.9) ---
torch.manual_seed(42)
result = top_p_sampling(lm, start, p=0.9, max_length=12)
results_comparison['Top-p(0.9)'] = lm.decode(result)
print(f'Nucleus (p=0.9):     {lm.decode(result)}')

# --- 6. Top-p + Temperature + Repetition Penalty ---
torch.manual_seed(42)
result = sample_with_penalties(lm, start, max_length=12,
                                temperature=0.8, top_p=0.9,
                                frequency_penalty=1.0, presence_penalty=0.5)
results_comparison['Combined'] = lm.decode(result)
print(f'Combined:            {lm.decode(result)}')

print()
print('→ 決定的手法（Greedy, Beam）は安定するが多様性に欠ける')
print('→ 確率的手法（Top-k, Top-p）は多様だが毎回異なる結果になる')
print('→ Combined（組み合わせ）が実用的にはよく使われる')

In [ ]:
# ============================================================
# セクション10（続き）: 多様性 vs 品質のトレードオフを可視化
# ============================================================
# 各手法で複数回生成し、多様性（ユニーク率）と品質（対数確率）
# のトレードオフを可視化する。
# ============================================================

def compute_diversity_score(model, start_tokens, generate_fn, n_samples=10, max_length=12):
    """
    多様性スコアと品質スコアを計算する。
    
    Returns:
        diversity: float — ユニークな生成テキストの割合
        avg_unique_ratio: float — 平均ユニークトークン率
    """
    texts = []
    unique_ratios = []
    
    for i in range(n_samples):
        torch.manual_seed(42 + i)
        result = generate_fn()
        text = model.decode(result)
        texts.append(text)
        
        # ユニークトークン率
        gen_tokens = result[len(start_tokens):]
        if len(gen_tokens) > 0:
            unique_ratios.append(len(set(gen_tokens)) / len(gen_tokens))
    
    diversity = len(set(texts)) / len(texts)
    avg_unique = np.mean(unique_ratios) if unique_ratios else 0
    return diversity, avg_unique

# 各手法の多様性を計算
methods = {
    'Greedy': lambda: greedy_search(lm, start, max_length=12)[0],
    'Beam(k=5)': lambda: beam_search(lm, start, beam_width=5, max_length=12)[0],
    'Temp(0.5)': lambda: sample_with_temperature(lm, start, temperature=0.5, max_length=12),
    'Temp(1.0)': lambda: sample_with_temperature(lm, start, temperature=1.0, max_length=12),
    'Temp(1.5)': lambda: sample_with_temperature(lm, start, temperature=1.5, max_length=12),
    'Top-k(5)': lambda: top_k_sampling(lm, start, k=5, max_length=12),
    'Top-k(20)': lambda: top_k_sampling(lm, start, k=20, max_length=12),
    'Top-p(0.5)': lambda: top_p_sampling(lm, start, p=0.5, max_length=12),
    'Top-p(0.9)': lambda: top_p_sampling(lm, start, p=0.9, max_length=12),
}

diversities = []
unique_ratios = []
method_names = []

for name, gen_fn in methods.items():
    div, uniq = compute_diversity_score(lm, start, gen_fn, n_samples=15, max_length=12)
    diversities.append(div)
    unique_ratios.append(uniq)
    method_names.append(name)

# --- 可視化 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('デコーディング戦略の比較: 多様性スコア', fontsize=14, fontweight='bold')

# テキストレベルの多様性（ユニークテキスト率）
colors = sns.color_palette('viridis', len(method_names))
ax1.barh(method_names, diversities, color=colors)
ax1.set_xlabel('テキスト多様性（ユニークテキスト率）')
ax1.set_title('テキストレベルの多様性')
ax1.set_xlim(0, 1.1)
for i, v in enumerate(diversities):
    ax1.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)

# トークンレベルの多様性（平均ユニークトークン率）
ax2.barh(method_names, unique_ratios, color=colors)
ax2.set_xlabel('トークン多様性（平均ユニークトークン率）')
ax2.set_title('トークンレベルの多様性')
ax2.set_xlim(0, 1.1)
for i, v in enumerate(unique_ratios):
    ax2.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('テキスト多様性: 生成されたテキストのうちユニークなものの割合')
print('トークン多様性: 生成テキスト内のユニークトークンの割合（繰り返しの少なさ）')

In [ ]:
# ============================================================
# セクション10（続き）: 各手法の比較表
# ============================================================
# 各デコーディング戦略の特徴をまとめた比較表を出力する。
# ============================================================

print('=' * 80)
print('各デコーディング戦略の比較表')
print('=' * 80)
print()
print(f'{"手法":<22} {"種類":<10} {"主要パラメータ":<18} {"多様性":<8} {"品質安定性":<10} {"適用場面"}')
print('-' * 80)

table_data = [
    ('Greedy Search',      '決定的', 'なし',             '低',   '高',       '翻訳、要約'),
    ('Beam Search',        '決定的', 'beam_width',       '低〜中', '高',     '翻訳、要約、キャプション'),
    ('Temperature',        '確率的', 'temperature',      '調整可', '調整可', '創作、対話'),
    ('Top-k Sampling',     '確率的', 'k',                '中〜高', '中',     '対話、ストーリー生成'),
    ('Nucleus (Top-p)',    '確率的', 'p',                '中〜高', '中〜高', '汎用（推奨）'),
    ('反復ペナルティ',     '後処理', 'freq/pres penalty', '向上',   '向上',   '長文生成'),
]

for row in table_data:
    print(f'{row[0]:<20} {row[1]:<8} {row[2]:<18} {row[3]:<8} {row[4]:<10} {row[5]}')

print()
print('推奨: Top-p (0.9) + Temperature (0.7-0.9) + 反復ペナルティ の組み合わせ')

## 11. HuggingFace `model.generate()` との対応

HuggingFace Transformers の `model.generate()` メソッドは、上記の全ての戦略を引数で指定できます。

In [ ]:
# ============================================================
# セクション11: HuggingFace model.generate() との対応
# ============================================================
# transformers ライブラリの generate() メソッドで使える
# 引数と、本ノートブックで学んだ戦略の対応関係をまとめる。
# ============================================================

print('=' * 80)
print('HuggingFace model.generate() との対応表')
print('=' * 80)
print()
print(f'{"本ノートの手法":<24} {"generate() の引数":<30} {"デフォルト値":<15} {"説明"}')
print('-' * 80)

hf_mapping = [
    ('Greedy Search',       'do_sample=False',             'False',         'サンプリングなし → Greedy'),
    ('Beam Search',         'num_beams=k',                 '1',             'ビーム幅の指定'),
    ('Temperature',         'temperature=T',               '1.0',           '分布の鋭さを調整'),
    ('Top-k Sampling',      'top_k=k',                     '50',            '上位k個でフィルタ'),
    ('Nucleus (Top-p)',     'top_p=p',                     '1.0',           '累積確率pでフィルタ'),
    ('反復ペナルティ',       'repetition_penalty=r',         '1.0',           'r>1で繰り返し抑制'),
    ('長さペナルティ',       'length_penalty=α',             '1.0',           'Beam Search用'),
    ('サンプリング有効化',   'do_sample=True',              'False',         'Top-k/p使用時に必要'),
]

for row in hf_mapping:
    print(f'{row[0]:<22} {row[1]:<30} {row[2]:<15} {row[3]}')

print()
print('--- 使用例 ---')
print()
print('# Greedy Search（デフォルト）')
print('output = model.generate(input_ids, max_new_tokens=50)')
print()
print('# Beam Search')
print('output = model.generate(input_ids, max_new_tokens=50, num_beams=5)')
print()
print('# Nucleus Sampling（推奨設定）')
print('output = model.generate(')
print('    input_ids,')
print('    max_new_tokens=50,')
print('    do_sample=True,')
print('    temperature=0.8,')
print('    top_p=0.9,')
print('    repetition_penalty=1.2')
print(')')
print()
print('# Top-k + Top-p 組み合わせ')
print('output = model.generate(')
print('    input_ids,')
print('    max_new_tokens=50,')
print('    do_sample=True,')
print('    top_k=50,')
print('    top_p=0.95,')
print('    temperature=0.7')
print(')')

## 12. まとめ・チートシート・よくある間違い

### デコーディング戦略選択フローチャート

```
テキスト生成が必要
│
├── 正確性・忠実性が重要？（翻訳、要約、コード生成）
│   ├── YES → Beam Search (num_beams=4-6, length_penalty=0.6-1.0)
│   └── NO ↓
│
├── 多様性・創造性が重要？（対話、ストーリー、ブレスト）
│   ├── YES → Nucleus Sampling (top_p=0.9-0.95, temperature=0.8-1.2)
│   └── NO ↓
│
├── バランスが必要？（一般的なチャットボット）
│   └── → Top-p=0.9 + Temperature=0.7 + repetition_penalty=1.1
│
└── 長文生成？
    └── → 上記 + frequency_penalty + presence_penalty
```

### チートシート

| パラメータ | 低い値 | 高い値 |
|-----------|--------|--------|
| temperature | 決定的・保守的 | 多様・ランダム |
| top_k | 候補少ない・保守的 | 候補多い・多様 |
| top_p | 候補少ない・保守的 | 候補多い・多様 |
| repetition_penalty | 繰り返し許容 | 繰り返し抑制 |
| num_beams | 高速（Greedy相当） | 高品質・低速 |
| length_penalty | 短い出力を好む | 長い出力を好む |

### よくある間違い

**1. `do_sample=False` のまま `temperature` や `top_p` を設定する**
- HuggingFace では `do_sample=True` を明示しないとサンプリングが有効にならない
- temperature=0.8 を設定しても do_sample=False だと Greedy になる

**2. Top-k と Top-p を混同する**
- Top-k: **固定数**のトークンを残す → 分布の形状を考慮しない
- Top-p: **累積確率**に基づき動的にトークン数を決める → 分布に適応
- Top-p の方が一般に優れるとされる（Holtzman et al., 2020）

**3. 温度を高くしすぎる**
- T > 2.0 はほぼ一様分布になり、意味不明なテキストが生成される
- 通常は T=0.7〜1.2 の範囲で調整する
- T=0 は Greedy と同等だが、数値的に不安定になるので注意

## 13. 自己評価クイズ

In [ ]:
# ============================================================
# セクション13: 自己評価クイズ
# ============================================================
# 学習内容の理解度をチェックするためのクイズ。
# 各問題を考えてから回答セルを実行してください。
# ============================================================

quiz_data = [
    {
        'question': 'Q1. Greedy Search の最大の問題点は何ですか？',
        'options': [
            'A) 計算が遅い',
            'B) 局所最適に陥りやすく、繰り返しが発生する',
            'C) ランダムすぎて品質が低い',
            'D) ビーム幅の設定が難しい'
        ],
        'answer': 'B',
        'explanation': 'Greedy Search は各ステップで最大確率のトークンを選ぶため、'
                      '全体として最適でない経路に陥りやすく、同じパターンの繰り返しが発生しやすい。'
    },
    {
        'question': 'Q2. Temperature T=0.1 と T=2.0 の違いとして正しいのは？',
        'options': [
            'A) T=0.1 の方が多様な出力になる',
            'B) T=2.0 の方が決定的な出力になる',
            'C) T=0.1 の方が鋭い（ピーキーな）分布になる',
            'D) 温度は確率分布に影響しない'
        ],
        'answer': 'C',
        'explanation': 'T<1 は分布を鋭くし（最大確率トークンにさらに集中）、'
                      'T>1 は分布を平坦にする（よりランダムな選択）。'
    },
    {
        'question': 'Q3. Top-k Sampling と Nucleus (Top-p) Sampling の最も重要な違いは？',
        'options': [
            'A) Top-k は確率的、Top-p は決定的',
            'B) Top-k は固定数の候補、Top-p は累積確率に基づく動的な候補数',
            'C) Top-k の方が常に品質が高い',
            'D) Top-p は Beam Search の一種である'
        ],
        'answer': 'B',
        'explanation': 'Top-k は常にk個のトークンを候補にするが、'
                      'Top-p は分布の形状に応じて候補数が動的に変化する。'
                      '鋭い分布では少数、平坦な分布では多数のトークンが候補になる。'
    },
    {
        'question': 'Q4. HuggingFace の generate() で Nucleus Sampling を使うとき、'
                    '必ず設定が必要な引数は？',
        'options': [
            'A) num_beams',
            'B) do_sample=True',
            'C) max_length のみ',
            'D) repetition_penalty'
        ],
        'answer': 'B',
        'explanation': 'do_sample=True を設定しないと、top_p や temperature を指定しても '
                      'サンプリングが有効にならず、Greedy Searchが実行されてしまう。'
    },
    {
        'question': 'Q5. 反復ペナルティの「頻度ペナルティ」と「プレゼンスペナルティ」の違いは？',
        'options': [
            'A) 両者は同じものである',
            'B) 頻度ペナルティは出現回数に比例、プレゼンスペナルティは出現の有無のみで判定',
            'C) プレゼンスペナルティの方が常に強い',
            'D) 頻度ペナルティは Temperature の別名である'
        ],
        'answer': 'B',
        'explanation': '頻度ペナルティはトークンの出現回数に比例してロジットを減らす。'
                      'プレゼンスペナルティは出現したかどうか（0 or 1）のみで一律に減らす。'
                      '頻度ペナルティは多く出現するほど強くなる。'
    },
]

print('=' * 60)
print('自己評価クイズ（全5問）')
print('各問題を考えてから、下のセルの回答を確認してください。')
print('=' * 60)
print()

for i, q in enumerate(quiz_data):
    print(f'\n{q["question"]}')
    for opt in q['options']:
        print(f'  {opt}')

In [ ]:
# ============================================================
# セクション13（続き）: クイズの回答
# ============================================================

print('=' * 60)
print('回答と解説')
print('=' * 60)

for i, q in enumerate(quiz_data):
    print(f'\n{q["question"]}')
    print(f'  正解: {q["answer"]}')
    print(f'  解説: {q["explanation"]}')

print()
print('=' * 60)
print('お疲れ様でした！')
print('次のステップ: 実際の言語モデル（GPT-2など）で各戦略を試してみましょう。')
print('=' * 60)